In [3]:
import pandas as pd
import numpy as np

In [4]:
rmp_df = pd.read_csv('fake_rmp.csv', index_col = 0)
rmp_df.head()

,Dr. Gerber,Dr. Johnson,Dr. Craig,Dr. Zhu,Dr. Wang,Dr. Murphy,Dr. Sun,Dr. Eagan,Dr. Ness,Dr. Sudyanti,Dr. Zhang,Dr. Wasab,Dr. Diallo,Dr. Hernandez,Dr. King
Student_1,NaN,3.0,NaN,5.0,4.0,4.0,5.0,2.0,NaN,4.0,2.0,4.0,5.0,2.0,3.0
Student_2,NaN,4.0,5.0,5.0,5.0,4.0,4.0,5.0,NaN,4.0,4.0,3.0,5.0,1.0,4.0
Student_3,4.0,4.0,2.0,4.0,3.0,5.0,2.0,4.0,4.0,5.0,2.0,3.0,3.0,5.0,3.0
Student_4,2.0,5.0,5.0,5.0,5.0,2.0,4.0,3.0,5.0,4.0,5.0,4.0,4.0,4.0,5.0
Student_5,3.0,NaN,4.0,NaN,NaN,4.0,5.0,2.0,5.0,3.0,5.0,5.0,2.0,4.0,4.0


## Part 2(a)

In [19]:
def user_collab_filter(df, target_user, similarity='Cosine', k=5):
    # check if the user is in the dataframe
    if target_user not in df.index:
        print(f"Error: {target_user} not found in the dataset.")
        return None

    # check if the user has missing ratings
    if not df.loc[target_user].isna().any():
        print(f"Error: {target_user} has no missing ratings.")
        return None

    # compute each user's average rating in the original dataframe
    user_means = df.mean(axis=1, skipna=True)

    df_avg = df.copy()
    # compute the mean of each row and fill NaN values with the mean
    for idx, row in df_avg.iterrows():
        row_mean = row.mean()
        if pd.isna(row_mean):
            df_avg.loc[idx] = row.fillna(0)
        else:
            df_avg.loc[idx] = row.fillna(row_mean)

    # center each row
    df_centered = df_avg.copy()
    for idx, row in df_centered.iterrows():
        row_mean = row.mean()
        df_centered.loc[idx] -= row_mean

    # find similarity between each user
    similarity_series = pd.Series(index=df.index, dtype=float)
    target_user_row = df_centered.loc[target_user]
    x = target_user_row.to_numpy()
    for idx, row in df_centered.iterrows():
        y = row.to_numpy()
        if similarity == 'Cosine':
            denom = np.linalg.norm(x) * np.linalg.norm(y)
            similarity_score = 0.0 if denom == 0 else np.dot(x, y) / denom
        elif similarity == 'L2':
            similarity_score = -np.linalg.norm(x - y)
        else:
            print(f"Error: {similarity} is not a valid similarity. Please use Cosine or L2.")
            return None
        similarity_series.loc[idx] = similarity_score

    # drop the user's similarity score to themselves
    similarity_series = similarity_series.drop(target_user)

    # min-max scale the similarity series
    similarity_min_max = (similarity_series - similarity_series.min()) / (similarity_series.max() - similarity_series.min())

    # find the k most similar users
    similarity_kmost = similarity_min_max.nlargest(k)
    users_kmost = similarity_kmost.index

    # find empty ratings for the target user
    empty_item_list = df.columns[df.loc[target_user].isna()].tolist()

    # predict ratings using k most similar users
    predicted_rating = {}
    for item in empty_item_list:
        similar_ratings = df.loc[users_kmost, item]
        similar_ratings = similar_ratings.fillna(user_means.loc[users_kmost])

        num = (similarity_kmost * similar_ratings).sum()
        denom = similarity_kmost.sum()
        
        if denom == 0:
            predicted_rating[item] = float(np.nan)
        else:
            predicted_rating[item] = float(num / denom)

    return predicted_rating

## Part 2(b)

In [20]:
user_collab_filter(rmp_df, "Student_5", similarity='Cosine', k=3)

{'Dr. Johnson': 4.14962596753536,
 'Dr. Zhu': 3.6693060979782794,
 'Dr. Wang': 3.6347751229366834}

In [21]:
user_collab_filter(rmp_df, "Student_3", similarity='Cosine', k=3)

Error: Student_3 has no missing ratings.


## Part 2(c)

In [24]:
print(f'Student 1 (cosine): {user_collab_filter(rmp_df, "Student_1", similarity='Cosine', k=5)['Dr. Gerber']}')
print(f'Student 1 (l2): {user_collab_filter(rmp_df, "Student_1", similarity='L2', k=5)['Dr. Gerber']}')

Student 1 (cosine): 4.396127956485075
Student 1 (l2): 3.826632123057846


Based on the computed ratings above, Student 1 would most likely enjoy Dr. Gerber's class. The student would likely give Dr. Gerber a round a 4/5, a solid rating. I would recommend this student to take Dr. Gerber.

In [26]:
print(f'Student 3 (cosine): {user_collab_filter(rmp_df, "Student_2", similarity='Cosine', k=5)['Dr. Gerber']}')
print(f'Student 3 (l2): {user_collab_filter(rmp_df, "Student_2", similarity='L2', k=5)['Dr. Gerber']}')

Student 3 (cosine): 4.251769420851984
Student 3 (l2): 4.56269731203369


Based on the computed ratings above, Student 2 would very likely enjoy Dr. Gerber's class. The student would likely give Dr. Gerber a round a 4.5/5, a strong rating. I would strongly recommend this student to take Dr. Gerber.

In [27]:
print(f'Student 13 (cosine): {user_collab_filter(rmp_df, "Student_13", similarity='Cosine', k=5)['Dr. Gerber']}')
print(f'Student 13 (l2): {user_collab_filter(rmp_df, "Student_13", similarity='L2', k=5)['Dr. Gerber']}')

Student 13 (cosine): 3.4335028612620593
Student 13 (l2): 4.174035557408685


Based on the computed ratings above, Student 13 would likely enjoy Dr. Gerber's class. The student would likely give Dr. Gerber a round a 3.75/5, a decent rating. I would recommend this student to take Dr. Gerber.

## Part 4(a)

In [29]:
def item_collab_filter(df, target_user, similarity='Cosine', k=5):
    # check if the user is in the dataframe
    if target_user not in df.index:
        print(f"Error: {target_user} not found in the dataset.")
        return None

    # check if the user has missing ratings
    if not df.loc[target_user].isna().any():
        print(f"Error: {target_user} has no missing ratings.")
        return None

    # compute each item's average rating in the original dataframe
    item_means = df.mean(axis=0, skipna=True)

    df_centered = df.copy()
    # center each column
    for col in df_centered.columns:
        col_mean = df_centered[col].mean()
        df_centered[col] = df_centered[col] - col_mean

    # find empty ratings for the target user
    empty_item_list = df.columns[df.loc[target_user].isna()].tolist()

    # predict ratings using k most similar items
    predicted_rating = {}
    for item in empty_item_list:
        # find similarity between each item
        similarity_series = pd.Series(index=df.columns, dtype=float)
        x_full = df_centered[item]

        for col in df_centered.columns:
            y_full = df_centered[col]

            corated_mask = x_full.notna() & y_full.notna()
            x = x_full[corated_mask].to_numpy()
            y = y_full[corated_mask].to_numpy()

            if len(x) == 0:
                similarity_score = 0.0
            elif similarity == 'Cosine':
                denom = np.linalg.norm(x) * np.linalg.norm(y)
                similarity_score = 0.0 if denom == 0 else np.dot(x, y) / denom
            elif similarity == 'L2':
                similarity_score = -np.linalg.norm(x - y)
            else:
                print(f"Error: {similarity} is not a valid similarity. Please use Cosine or L2.")
                return None

            similarity_series.loc[col] = similarity_score

        # drop the item's similarity score to itself
        similarity_series = similarity_series.drop(item)

        # min-max scale the similarity series
        similarity_min_max = (similarity_series - similarity_series.min()) / (similarity_series.max() - similarity_series.min())

        # find the k most similar items
        similarity_kmost = similarity_min_max.nlargest(k)
        users_kmost = similarity_kmost.index

        similar_ratings = df.loc[target_user, users_kmost]
        similar_ratings = similar_ratings.fillna(item_means.loc[users_kmost])

        num = (similarity_kmost * similar_ratings).sum()
        denom = similarity_kmost.sum()

        if denom == 0:
            predicted_rating[item] = float(np.nan)
        else:
            predicted_rating[item] = float(num / denom)

    return predicted_rating

In [30]:
def collab_filter(filter_type, df, target_user, similarity='Cosine', k=5):
    if filter_type == 'User':
        return user_collab_filter(df, target_user, similarity, k)
    elif filter_type == 'Item':
        return item_collab_filter(df, target_user, similarity, k)
    else:
        print(f"Error: {filter_type} not found a valid filter. Please use User or Item.")
        return None

## Part 1

In [46]:
rating_table = pd.DataFrame(
    [
        [np.nan, 5, 3, 10, 7, 8],
        [7, np.nan, 6, 4, np.nan, 1],
        [5, 3, np.nan, 8, 3, 10],
        [6, 9, 6, np.nan, 1, 3],
    ],
    index=["User 1", "User 2", "User 3", "User 4"],
    columns=["Item 1", "Item 2", "Item 3", "Item 4", "Item 5", "Item 6"]
)

rating_table_original = rating_table.copy()

rating_table_filled = rating_table.copy()
for user in rating_table.index:
    preds = collab_filter("User", rating_table, user, similarity="Cosine", k=2)
    for item, val in preds.items():
        rating_table_filled.loc[user, item] = val

print("Original Table")
display(rating_table_original)

print("Predicted Table")
display(rating_table_filled)

Original Table


/var/folders/nm/jvl99pgd743bfkz7yrgq75s80000gn/T/ipykernel_74415/3442526346.py:28: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1.3999999999999995' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_centered.loc[idx] -= row_mean
/var/folders/nm/jvl99pgd743bfkz7yrgq75s80000gn/T/ipykernel_74415/3442526346.py:28: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1.3999999999999995' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_centered.loc[idx] -= row_mean
/var/folders/nm/jvl99pgd743bfkz7yrgq75s80000gn/T/ipykernel_74415/3442526346.py:28: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1.3999999999999995' has dtype incompatible with int64, please explicitly cast to

,Item 1,Item 2,Item 3,Item 4,Item 5,Item 6
User 1,NaN,5.0,3.0,10.0,7.0,8
User 2,7.0,NaN,6.0,4.0,NaN,1
User 3,5.0,3.0,NaN,8.0,3.0,10
User 4,6.0,9.0,6.0,NaN,1.0,3


Predicted Table


,Item 1,Item 2,Item 3,Item 4,Item 5,Item 6
User 1,5.0506,5.000000,3.000000,10.000000,7.000000,8
User 2,7.0000,8.517392,6.000000,4.000000,1.723912,1
User 3,5.0000,3.000000,3.765774,8.000000,3.000000,10
User 4,6.0000,9.000000,6.000000,4.759858,1.000000,3


## Part 3(a)

In [45]:
rating_table = pd.DataFrame(
    [
        [np.nan, 5, 3, 10, 7, 8],
        [7, np.nan, 6, 4, np.nan, 1],
        [5, 3, np.nan, 8, 3, 10],
        [6, 9, 6, np.nan, 1, 3],
    ],
    index=["User 1", "User 2", "User 3", "User 4"],
    columns=["Item 1", "Item 2", "Item 3", "Item 4", "Item 5", "Item 6"]
)

rating_table_original = rating_table.copy()

rating_table_filled = rating_table.copy()
for user in rating_table.index:
    preds = collab_filter("Item", rating_table, user, similarity="Cosine", k=2)
    for item, val in preds.items():
        rating_table_filled.loc[user, item] = val

print("Original Table")
display(rating_table_original)

print("Predicted Table")
display(rating_table_filled)

Original Table


,Item 1,Item 2,Item 3,Item 4,Item 5,Item 6
User 1,NaN,5.0,3.0,10.0,7.0,8
User 2,7.0,NaN,6.0,4.0,NaN,1
User 3,5.0,3.0,NaN,8.0,3.0,10
User 4,6.0,9.0,6.0,NaN,1.0,3


Predicted Table


,Item 1,Item 2,Item 3,Item 4,Item 5,Item 6
User 1,3.974192,5.000000,3.000000,10.000000,7.000000,8
User 2,7.000000,6.501782,6.000000,4.000000,2.687642,1
User 3,5.000000,3.000000,4.028433,8.000000,3.000000,10
User 4,6.000000,9.000000,6.000000,1.980662,1.000000,3


## Part 3(b)

It's pretty difficult to discuss this as we don't have labels for true ratings. It's possible to, using the ratings we have, leave one out then compute the predicted rating for both item-item and user-user, and see which one is closer to the true rating. For scale and sparsity, there isn't an obvious answer either.

## Part 4(b)

In [31]:
collab_filter('Item', rmp_df, "Student_5", similarity='Cosine', k=2)

{'Dr. Johnson': 3.5429995715554776,
 'Dr. Zhu': 3.175814607755104,
 'Dr. Wang': 4.0}

## Part 4(c)

In [32]:
print(f'Student 1 (cosine): {collab_filter('Item', rmp_df, "Student_1", similarity='Cosine', k=5)['Dr. Gerber']}')
print(f'Student 1 (l2): {collab_filter('Item', rmp_df, "Student_1", similarity='L2', k=5)['Dr. Gerber']}')

Student 1 (cosine): 3.1637979648144694
Student 1 (l2): 2.900790299602153


Based on the computed ratings above, Student 1 might enjoy Dr. Gerber's class. The student would likely give Dr. Gerber a round a 3/5, a half decent rating. I would recommend this student consider Dr. Gerber's class, but not strongly recommend it.

In [36]:
print(f'Student 2 (cosine): {collab_filter('Item', rmp_df, "Student_2", similarity='Cosine', k=5)['Dr. Gerber']}')
print(f'Student 2 (l2): {collab_filter('Item', rmp_df, "Student_2", similarity='L2', k=5)['Dr. Gerber']}')

Student 2 (cosine): 3.7093325512200006
Student 2 (l2): 3.7009378266990813


Based on the computed ratings above, Student 2 would likely enjoy Dr. Gerber's class. The student would likely give Dr. Gerber a round a 3.7/5, a good rating. I would recommend this student to take Dr. Gerber.

In [37]:
print(f'Student 13 (cosine): {collab_filter('Item', rmp_df, "Student_13", similarity='Cosine', k=5)['Dr. Gerber']}')
print(f'Student 13 (l2): {collab_filter('Item', rmp_df, "Student_13", similarity='L2', k=5)['Dr. Gerber']}')

Student 13 (cosine): 3.597725975305312
Student 13 (l2): 4.163361855951449


Based on the computed ratings above, Student 13 would very likely enjoy Dr. Gerber's class. The student would likely give Dr. Gerber a round a 4/5, a strong rating. I would highly recommend this student to take Dr. Gerber.

## Part 5

While this is not a complete data set, here is a link to one reviewer's ratings of Wollaston's sandwiches: https://www.reddit.com/r/NEU/comments/w9nuo4/ratingreviewing_all_wollastons_sandwiches_that/

My idea would be to set up a website where users can rate sandiches out of 10 from Wollastons @ Northeastern. This would create a database of users who buy and rate Wollastons's sandwiches and their ratings of each sandwiches. This is perfect for collaborative filtering. A problem I and a lot of students/faculty have is deciding which sandich to get out of the many options. Instead of trying each sandwich, we can use collaboriate filtering to asses whether a user would like a sandwich. This would likely use item based filtering as, like Amazon, scale and sparsity of the database would favor item based filtering.

One big ethical concern is if users come together to tilt the ratings to favor certain sandiches. Frats and organizations have their signature sandiwiches. While I'm not sure if they receive any money from people buying their sandwiches, organizations may come together to skew the ratings to incentivise people to buy their sandiwches more often. This would make the collaborative filtering process dishonest and exploitable.